# **Summarizing Private Documents Using RAG, LangChain, and Ollama**

A hands-on notebook for building a fully local, private RAG pipeline — no API keys, no data sent to the cloud. Inspired by a RAG tutorial originally built on a cloud LLM platform, reimplemented here using an open-source local stack.

## Background

### What is RAG?
Retrieval-Augmented Generation (RAG) augments an LLM's knowledge with your own private data. Instead of uploading sensitive documents to an external service, the document is embedded locally into a vector store and the LLM queries it at runtime — nothing leaves your machine.

### RAG Architecture
**Indexing**
1. **Load** — read your document
2. **Split** — break it into overlapping chunks
3. **Embed & Store** — convert chunks to vectors and store in ChromaDB

**Retrieval & Generation**
1. **Retrieve** — find the most relevant chunks for a query
2. **Generate** — pass query + chunks to the local LLM for an answer


## Setup

### Step 1 — Install Ollama and pull Gemma 2 2B

Run these once in a terminal (not in the notebook):

```bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Pull Gemma 2 2B (~1.6 GB download)
ollama pull gemma2:2b

# Verify it works
ollama run gemma2:2b "hello"
```

Ollama runs as a background service on `http://localhost:11434`. LangChain talks to it via HTTP — no Python bindings needed for the model itself.

### Step 2 — Install Python dependencies

In [18]:
%%capture
!pip install langchain==0.3.25
!pip install langchain-community==0.3.24
!pip install langchain-ollama==0.3.3
!pip install langchain-huggingface==0.1.2
!pip install langchain-chroma==0.2.4
!pip install sentence-transformers==3.4.1
!pip install chromadb>=1.0.9
!pip install pypdf>=4.0.0
!pip install wget==3.2

**Restart the kernel after installation**, then continue from the next cell.

### Step 3 — Start Ollama server

In [ ]:
import subprocess
import time
import urllib.request
import os

OLLAMA_BIN = os.path.expanduser("~/.local/bin/ollama")

def is_ollama_running():
    try:
        urllib.request.urlopen("http://localhost:11434")
        return True
    except:
        return False

if is_ollama_running():
    print("Ollama already running.")
else:
    print("Starting Ollama server...")
    env = os.environ.copy()
    env["OLLAMA_MODELS"] = os.path.expanduser("~/.ollama/models")
    env["OLLAMA_CONTEXT_LENGTH"] = "2048"   # limit context to prevent OOM
    subprocess.Popen(
        [OLLAMA_BIN, "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    for _ in range(15):
        time.sleep(1)
        if is_ollama_running():
            print("Ollama server started successfully.")
            break
    else:
        print(f"ERROR: Could not start Ollama. Check that {OLLAMA_BIN} exists.")

In [20]:
import urllib.request
try:
    with urllib.request.urlopen("http://localhost:11434") as r:
        print("Ollama is running:", r.read().decode())
except Exception as e:
    print("Ollama not reachable:", e)

Ollama is running: Ollama is running


### Step 4 — Import all libraries

In [21]:
import warnings
warnings.filterwarnings('ignore')

# Document loading and splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter

# Embeddings and vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# LLM — Mistral 7B via Ollama (local, no API key)
from langchain_ollama import OllamaLLM

# Chains
from langchain.chains import RetrievalQA
from langchain.chains import ConversationalRetrievalChain

# Prompt and memory
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory

import wget

print("All imports OK")

All imports OK


---
## Preprocessing

### Load the document

`PyPDFLoader` parses the PDF and extracts text page by page. Each page becomes a separate `Document` object. This is the **Load** step of Indexing.

In [22]:
filename = 'brick-klins-paper.pdf'
url = 'https://openreview.net/pdf?id=efGzsxVSEC'

wget.download(url, out=filename)
print('\nFile downloaded:', filename)


File downloaded: brick-klins-paper.pdf


In [23]:
loader = PyPDFLoader(filename)
documents = loader.load()

print(f"PDF loaded: {len(documents)} pages")
print("\n--- Page 1 preview ---")
print(documents[0].page_content[:1000])

PDF loaded: 29 pages

--- Page 1 preview ---
SENTINEL KILN DB: A Large-Scale Dataset and
Benchmark for OBB Brick Kiln Detection in South
Asia Using Satellite Imagery
Rishabh Mondal1, Jeet Parab2, Heer Kubadia1, Shataxi Dubey1, Shardul Junagade1, Zeel B Patel1,
Nipun Batra1
1IIT Gandhinagar, India, 2IIIT Surat, India
Abstract
Air pollution was responsible for 2.6 million deaths across South Asia in 2021 alone,
with brick manufacturing contributing significantly to this burden. In particular,
the Indo-Gangetic Plain; a densely populated and highly polluted region spanning
northern India, Pakistan, Bangladesh, and parts of Afghanistan sees brick kilns con-
tributing 8–14% of ambient air pollution. Traditional monitoring approaches, such
as field surveys and manual annotation using tools like Google Earth Pro, are time
and labor-intensive. Prior ML-based efforts for automated detection have relied
on costly high-resolution commercial imagery and non-public datasets, limiting
reproducibilit

### Split the document into chunks

This is the **Split** step. `CharacterTextSplitter` breaks the pages into chunks of ~1000 characters. Each chunk becomes one independently searchable unit.

In [24]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

print(f"Document split into {len(texts)} chunks")

Document split into 29 chunks


### Embed and store in ChromaDB

This is the **Embed & Store** step. `HuggingFaceEmbeddings` uses `all-MiniLM-L6-v2` (downloads ~90 MB once, runs locally). Each chunk is converted to a 384-dimension vector and stored in an in-memory ChromaDB collection.

In [25]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

docsearch = Chroma.from_documents(texts, embeddings)
print("Document embedded and stored in ChromaDB")

Document embedded and stored in ChromaDB


---
## LLM Construction — Gemma 2 2B via Ollama

`OllamaLLM` sends requests to the local Ollama server (`http://localhost:11434`). The model runs on your CPU — no API key, no data sent externally.

In [37]:
llm = OllamaLLM(
    model="gemma2:2b",
    temperature=0.5,
    num_predict=256,
    num_ctx=2048,       # cap context window to prevent OOM on low RAM
    top_k=40,
    top_p=0.9,
)

print(llm.invoke("Say hello in one sentence."))

Hello! 



---
## Integrating LangChain — RetrievalQA

LangChain's `RetrievalQA` ties together the vector store retriever and the LLM. For each query:
1. ChromaDB finds the top-k most similar chunks
2. They are stuffed into the prompt context
3. Gemma 2 generates an answer

### Ask a specific question

In [27]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    return_source_documents=False
)

query = "What is the main contribution of this paper?"
result = qa.invoke(query)
print(result["result"])

This paper discusses a novel approach to tackling air pollution using machine learning models.  It details the methodology and presents experimental results that show promising potential for real-world applications. 


**Please note:** I cannot definitively state the exact main contribution without more context or specific information about the paper's content. 



### Ask a high-level summary question

In [28]:
query = "Can you summarize the document for me?"
result = qa.invoke(query)
print(result["result"])

This NeurIPS paper explores a novel approach to address air pollution concerns, focusing on the use of large language models (LLMs) to generate creative solutions and strategies. The authors detail their methods, experimental results, and limitations in their research.  

Here are some key points:

* **Focus:** Utilizing LLMs for air pollution mitigation
* **Methodology:** The paper describes how they employed LLMs to generate innovative solutions and strategies to address air pollution concerns. 
* **Data & Resources:** The authors provide access to the code, data, and instructions needed to reproduce their experiments, ensuring reproducibility of results.  They also discuss limitations and potential societal impacts. 

**Overall**: The paper offers a promising approach for tackling air pollution challenges through the use of LLMs, with clear steps towards reproducible research and ethical considerations. 



---
## Dive Deeper

### Using a Prompt Template

Without a system prompt, the LLM may hallucinate answers for questions not in the document. The prompt below instructs it to say "I don't know" instead of guessing.

In [29]:
prompt_template = """Use only the information from the document below to answer the question.
If the answer is not in the document, say "I don't know" — do not make up an answer.

{context}

Question: {question}
Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [30]:
qa_with_prompt = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    chain_type_kwargs=chain_type_kwargs,
    return_source_documents=False
)

query = "how are brick-klins detected"
result = qa_with_prompt.invoke(query)
print(result["result"])

I don't know 



### Make the Conversation Have Memory

`ConversationBufferMemory` keeps the full chat history in memory. `ConversationalRetrievalChain` uses it to resolve follow-up questions like "What about *it*?" where "it" refers to something mentioned earlier.

In [31]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

qa_memory = ConversationalRetrievalChain.from_llm(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    memory=memory,
    return_source_documents=False
)

In [32]:
history = []

query = "What is the main topic of this document?"
result = qa_memory.invoke({"question": query, "chat_history": history})
print(result["answer"])

The main topic of this document is a NeurIPS paper checklist. 



In [33]:
history.append((query, result["answer"]))

query = "List the key points in it."
result = qa_memory.invoke({"question": query, "chat_history": history})
print(result["answer"])

This NeurIPS paper checklist outlines the necessary components for a research paper submitted to the conference.  Here's a breakdown based on your provided context: 

**Key Points:**

* **Accuracy & Scope:** The abstract and introduction clearly represent the paper's contributions and scope.
* **Limitations:** The paper acknowledges limitations of the work done by the authors.
* **Theoretical Foundation:** The paper provides the assumptions for theoretical results, and attempts to provide proofs. 
* **Reproducibility:** The paper details how to reproduce the experiments in a format that allows others to do so (code, data, instructions). 
* **Data & Code Access:**  The research team has made their data and code publicly available.
* **Experimental Details:** The paper provides sufficient information on training and test details for understanding results. 
* **Statistical Significance:** The paper reports error bars, but also mentions the statistical significance of experiments. 
* **Com

In [34]:
history.append((query, result["answer"]))

query = "What is the aim of it?"
result = qa_memory.invoke({"question": query, "chat_history": history})
print(result["answer"])

The purpose of this NeurIPS paper checklist is to ensure that a submitted paper adheres to ethical and methodological standards set by the conference. It covers various aspects like claims, limitations, theoretical assumptions, reproducibility, open access to data and code, experimental settings, statistical significance, compute resources, ethical considerations (code of ethics), broader impacts, safeguards for data or models with high risk of misuse, proper attribution and documentation for existing assets, new assets, crowdsourcing and research with human subjects, IRB approvals, LLM usage. 



### Wrap Up — Interactive Agent

The function below creates a fresh conversation session with memory each time it's called. Type `quit`, `exit`, or `bye` to stop.

In [38]:
def run_agent():
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        chain_type="stuff",
        retriever=docsearch.as_retriever(),
        memory=memory,
        return_source_documents=False
    )
    history = []
    print("Agent ready. Ask anything about the document. Type 'quit' to exit.\n")

    while True:
        query = input("You: ")
        if query.lower() in ["quit", "exit", "bye"]:
            print("Agent: Goodbye!")
            break
        result = chain.invoke({"question": query, "chat_history": history})
        history.append((query, result["answer"]))
        print(f"Agent: {result['answer']}\n")

run_agent()

Agent ready. Ask anything about the document. Type 'quit' to exit.



KeyboardInterrupt: 

---
## Exercises

### Exercise 1: Use your own PDF

Point `PyPDFLoader` at any PDF on your machine — research papers, reports, contracts. Nothing is sent to any server.

In [ ]:
# Change this to any PDF path on your machine
my_pdf = 'your_document.pdf'

loader2 = PyPDFLoader(my_pdf)
docs2 = loader2.load()
texts2 = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(docs2)
docsearch2 = Chroma.from_documents(texts2, embeddings)

qa2 = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch2.as_retriever(),
    return_source_documents=False
)
print(qa2.invoke("What is the main topic?")["result"])

### Exercise 2: Return source chunks alongside the answer

In [ ]:
qa_src = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    return_source_documents=True
)

query = "What methods are used in this paper?"
result = qa_src.invoke(query)

print("Answer:", result["result"])
print("\n--- Source chunk ---")
print(result["source_documents"][0].page_content)

### Exercise 3: Try a different local model

Swap Mistral for any model available in Ollama. Pull it first in a terminal, then change the `model=` argument:

```bash
ollama pull llama3.2        # 2 GB, fast
ollama pull phi3            # 2.2 GB, strong reasoning
```

In [ ]:
llm_alt = OllamaLLM(
    model="mistral",
    temperature=0.5,
    num_predict=256,
)

qa_alt = RetrievalQA.from_chain_type(
    llm=llm_alt,
    chain_type="stuff",
    retriever=docsearch.as_retriever(),
    return_source_documents=False
)

print(qa_alt.invoke("Can you summarize the document?")["result"])